In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2014great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2014_psysc_LT_edited.csv")
complete_path_2 = os.path.join(original_data_pathway, "2014_psysc_prop.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
# df1 = df1.assign(experiment_name='LT')
# df1 = df1.assign(experiment='1')
df1.rename(columns={"subject": "ape",
    "sex":"sex_original"}, inplace=True)
df1=df1.applymap(lambda s: s.lower() if type(s) == str else s)
df1['study_id']="kano2014great"

df2 = pd.read_csv(complete_path_2)
# df2 = df2.assign(experiment_name='prop')
# df2 = df2.assign(experiment='2')

df2_1 = df2[['subject', 'Hand', 'Claw']]


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df1['ape'] = df1['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df1['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df1= df1.merge(apedf,left_on='ape', right_on='name', how='left')
# df2.columns

In [4]:
data_frames=[df1, df2_1]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    # x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    # x.rename(columns={"subject": "ape",
        # "sex":"sex_original"}, inplace=True)
    # x['ape'] = x['ape'].str.rstrip()
    # x['study_id']="kano2014great"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, axis=1)


In [5]:
fulldf.columns = fulldf.columns.str.replace(' ', '_')
import re
replace_1=re.compile('(\(|\)|\*)') 
fulldf.columns = fulldf.columns.str.replace(replace_1, '')
fulldf.rename(columns={"ape": "participant",
    "target_famreach__hand":'target_famreach_hand',
    'age':'age_original'}, inplace=True)


changes = ['target_famreach_hand',
       'target_famreach_claw', 'target_grasp_hand', 'target_grasp_claw',
       'target_testreach_hand', 'target_testreach_claw',
       'distractor_famreach_hand', 'distractor_famreach_claw',
       'distractor_grasp_hand', 'distractor_grasp_claw',
       'distractor_testreach_hand', 'distractor_testreach_claw']
for x in changes:
    fulldf.columns = fulldf.columns.str.replace(x, x+'_in_seconds')
# fulldf.columns

In [6]:

kano2014great_standardized=fulldf[['study_id', 'participant', 'age_original', 'sex',
       'species','hand', 'claw', 'target_famreach_hand_in_seconds',
       'target_famreach_claw_in_seconds', 'target_grasp_hand_in_seconds', 'target_grasp_claw_in_seconds',
       'target_testreach_hand_in_seconds', 'target_testreach_claw_in_seconds',
       'distractor_famreach_hand_in_seconds', 'distractor_famreach_claw_in_seconds',
       'distractor_grasp_hand_in_seconds', 'distractor_grasp_claw_in_seconds',
       'distractor_testreach_hand_in_seconds', 'distractor_testreach_claw_in_seconds']]
comp_out_path_stand = os.path.join(out_pathway, 'kano2014great_standardized.csv')
kano2014great_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =kano2014great_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kano2014great_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kano2014great_glossary.csv')
kano2014great_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
